# Pydantic

### Pydantic models provide the richest feature set with field validation, descriptions and nested structures

In [ ]:
# Step-1: Initialise the environment
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [ ]:
# Step-2: Integrate Chat model
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x000002D9641DF4D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002D9641DFED0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
# Step-3: Make a structured class for making Pydantic model

from pydantic import Field
from pydantic import BaseModel

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The movie was released in this year")
    rating:float=Field(description="The rating of the movie")
    director:str=Field(description="The movie was directed by this director")


In [ ]:
# Step 4: Binding model to class
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x000002D963D6FCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002D963F90AD0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The movie was released in this year', 'type': 'integer'}, 'rating': {'description': 'The rating of the movie', 'type': 'number'}, 'director': {'description': 'The movie was directed by this director', 'type': 'string'}}, 'required': ['title', 'year', 'rating', 'director'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': 

In [ ]:
# Step 5: Generating Response 
response = model_with_structure.invoke("Provide details about the movie RRR")
response

Movie(title='RRR', year=2022, rating=7.8, director='S. S. Rajamouli')

In [ ]:
# You can see both raw as well as parsed messages by "include_raw = True"
from pydantic import Field
from pydantic import BaseModel

class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The movie was released in this year")
    rating:float=Field(...,description="The rating of the movie")
    director:str=Field(...,description="The movie was directed by this director")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure

response = model_with_structure.invoke("Provide details about the movie RRR")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2rddaxq5y', 'function': {'arguments': '{"director":"S. S. Rajamouli","rating":8.2,"title":"RRR","year":2022}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 349, 'total_tokens': 423, 'completion_time': 0.20017316, 'completion_tokens_details': None, 'prompt_time': 0.028881618, 'prompt_tokens_details': None, 'queue_time': 0.064614811, 'total_time': 0.229054778}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0edc1-fe0a-72d3-aed5-2e030c5d8bd0-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'S. S. Rajamouli', 'rating': 8.2, 'title': 'RRR', 'year': 2022}, 'id': '2rddaxq5y', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 349, 'output_tokens': 74, 'total_tokens': 423}),


# Nested Structure

In [35]:
class Actor(BaseModel):
    name:str
    role:str

class Movie_Details(BaseModel):
    title:str
    year:str
    actor:list[Actor]
    genre:list[str]
    budget:float | None = Field(None,description="The budget of the movie in USD")

model_with_structure = model.with_structured_output(Movie_Details)

#Using Invoke 
response = model_with_structure.invoke("Provide details about the movie Inception")
response

# # Using Streaming
 
# for chunk in model_with_structure.stream("Provide details about the movie Inception"):
#     print(chunk.text,end = "", flush=True)


Movie_Details(title='Inception', year='2010', actor=[Actor(name='Leonardo DiCaprio', role='Dom Cobb')], genre=['Action', 'Science Fiction', 'Thriller'], budget=None)

# TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The movie was released in this year"]
    rating:Annotated[float,...,"The rating of the movie"]
    director:Annotated[str,...,"The movie was directed by this director"]

model_with_typedict=model.with_structured_output(MovieDict)
response=model_with_typedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [36]:
class Actor(TypedDict):
    name:str
    role:str

class Movie_Details(TypedDict):
    title:str
    year:str
    actor:list[Actor]
    genre:list[str]
    budget:float | None = Field(None,description="The budget of the movie in USD")

model_with_structure = model.with_structured_output(Movie_Details)

#Using Invoke 
response = model_with_structure.invoke("Provide details about the movie Inception")
response


{'actor': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'}],
 'budget': 160000000,
 'genre': ['Action', 'Science Fiction', 'Thriller'],
 'title': 'Inception',
 'year': '2010'}

In [39]:
model.profile

# Dataclasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator


Making Pydantic agents ⬇️

In [41]:
import os   
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
from pydantic import Field,BaseModel
from langchain.agents import create_agent

class Contact_Info(BaseModel):
    """Contact information for a person."""
    name:str=Field(...,description="The name of the person")
    email:str=Field(...,description="The email of the person")
    phone:str=Field(...,description="The phone number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    response_format=Contact_Info
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='3408cf33-5ee4-4a85-8f76-eceb16ed6a1c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wxghfmjqe', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'Contact_Info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 350, 'total_tokens': 415, 'completion_time': 0.214603618, 'completion_tokens_details': None, 'prompt_time': 0.02463475, 'prompt_tokens_details': None, 'queue_time': 0.060880659, 'total_time': 0.239238368}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee5c-6bab-7db2-80b2-8401be57dd15-0', tool_calls=[{'name': 'Contact_Info', 'args': {'email': 'john@ex

In [44]:
result["structured_response"]

Contact_Info(name='John Doe', email='john@example.com', phone='(555) 123-4567')

Making Typedict agents⬇️

In [ ]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class Contact_Info(TypedDict):
    """Contact information for a person."""
    name:str
    email:str
    phone:str

agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    response_format=Contact_Info
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

Making agent using Dataclasses ⬇️

In [49]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contact_Info:
    """Contact information for a person."""
    name:str
    email:str
    phone:str

agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    response_format=Contact_Info
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

Contact_Info(name='John Doe', email='john@example.com', phone='(555) 123-4567')